In [ ]:
!pip install pandas openpyxl
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sqlite3

# Conectar ao banco de dados Adendos_Tabelado.db
conn_adendos = sqlite3.connect('/content/drive/MyDrive/PRIVATE_DATA/Adendos_Tabelado.db')

# Conectar ao banco de dados relatorio_de_alta_tabelada.db
conn_relatorios = sqlite3.connect('/content/drive/MyDrive/PRIVATE_DATA/relatorio_de_alta_tabelada.db')

# Listar as tabelas presentes em cada banco de dados
adendos_tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn_adendos)
relatorios_tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn_relatorios)

adendos_df = pd.read_sql_query("SELECT * FROM adendos_tabelados;", conn_adendos)
relatorios_df = pd.read_sql_query("SELECT * FROM df_notas_clinicas;", conn_relatorios)

# Fechar conexões de banco de dados
conn_adendos.close()
conn_relatorios.close()


In [ ]:
print(adendos_df,relatorios_df)

# Limpando os Adendos
Contando quantos adendos começam em cada um desses anos

In [ ]:
count_2019 = adendos_df['Nome do Arquivo'].str.startswith('2019').sum()
count_2020 = adendos_df['Nome do Arquivo'].str.startswith('2020').sum()
count_2021 = adendos_df['Nome do Arquivo'].str.startswith('2021').sum()
count_2022 = adendos_df['Nome do Arquivo'].str.startswith('2022').sum()
count_2023 = adendos_df['Nome do Arquivo'].str.startswith('2023').sum()

print(f"Quantos começam com 2019: {count_2019}")
print(f"Quantos começam com 2020: {count_2020}")
print(f"Quantos começam com 2021: {count_2021}")
print(f"Quantos começam com 2022: {count_2022}")
print(f"Quantos começam com 2023: {count_2023}")

In [ ]:
linhas_nao_comecam_com_anos = adendos_df[~adendos_df['Nome do Arquivo'].str.startswith(('2019', '2020', '2021', '2022', '2023'))]
linhas_nao_comecam_com_anos

Vou tirar as linhas que não fazem sentido manualmente

In [ ]:
adendos_df = adendos_df.drop([157, 435, 460, 520, 1155, 1164, 1165, 1200, 1826, 4307, 4417, 4418])


Removendo o que começa com 2019

In [ ]:
adendos_df = adendos_df[~adendos_df['Nome do Arquivo'].str.startswith('2019')]
adendos_df

#Limpando os Relatórios de Alta

In [ ]:
count_2019 = relatorios_df['Nome do Arquivo'].str.startswith('2019').sum()
count_2020 = relatorios_df['Nome do Arquivo'].str.startswith('2020').sum()
count_2021 = relatorios_df['Nome do Arquivo'].str.startswith('2021').sum()
count_2022 = relatorios_df['Nome do Arquivo'].str.startswith('2022').sum()
count_2023 = relatorios_df['Nome do Arquivo'].str.startswith('2023').sum()

print(f"Quantos começam com 2019: {count_2019}")
print(f"Quantos começam com 2020: {count_2020}")
print(f"Quantos começam com 2021: {count_2021}")
print(f"Quantos começam com 2022: {count_2022}")
print(f"Quantos começam com 2023: {count_2023}")

In [ ]:
linhas_nao_comecam_com_anos = adendos_df[~adendos_df['Nome do Arquivo'].str.startswith(('2019', '2020', '2021', '2022', '2023'))]
linhas_nao_comecam_com_anos

Não vi problema grave em nenhum dos anteriores, então só vou excluir os de 2019

In [ ]:
relatorios_df = relatorios_df[~relatorios_df['Nome do Arquivo'].str.startswith('2019')]
relatorios_df

Vou transformar em um arquivo_db, para fazer uma analise exploratória melhor, e alguns ajustes nos dados


In [ ]:
# Criar uma conexão com o banco de dados SQLite (ou criar um novo se não existir)
conn = sqlite3.connect('meu_banco_de_dados.db')

# Salvar o DataFrame adendos_df na tabela 'adendos'
adendos_df.to_sql('adendos', conn, if_exists='replace', index=False)

# Salvar o DataFrame relatorios_df na tabela 'relatorios'
relatorios_df.to_sql('relatorios', conn, if_exists='replace', index=False)

# Fechar a conexão com o banco de dados
conn.close()

##Fiz algumas Correções Manuais no SQLITe agora estou puxando o corrigido

In [ ]:
# Conectar ao banco de dados SQLite
caminho_db = '/content/drive/MyDrive/PRIVATE_DATA/Adendos_Relatorios_Separados_30_08.db'
conn = sqlite3.connect(caminho_db)

# Carregar a tabela 'adendos' em um DataFrame
adendos_df = pd.read_sql_query("SELECT * FROM adendos;", conn)

# Carregar a tabela 'relatorios' em um DataFrame
relatorios_df = pd.read_sql_query("SELECT * FROM relatorios;", conn)

# Fechar a conexão com o banco de dados
conn.close()


Formatando o nome


In [ ]:
relatorios_df

In [ ]:
adendos_df['Nome da Mãe'] = adendos_df['Nome da Mãe'].str.lower()
relatorios_df['Nome da Mãe'] = relatorios_df['Nome da Mãe'].str.lower()

In [ ]:
import string

# Criar uma tradução que remove pontuações
translator = str.maketrans('', '', string.punctuation)

# Remover pontuações na coluna 'Nome da Mãe' para ambos os DataFrames
adendos_df['Nome da Mãe'] = adendos_df['Nome da Mãe'].str.translate(translator)
relatorios_df['Nome da Mãe'] = relatorios_df['Nome da Mãe'].str.translate(translator)

print(adendos_df.shape, relatorios_df.shape)

In [ ]:
import re
import unicodedata

# Função para remover acentuação
def remover_acentuacao(texto):
    if isinstance(texto, str):
        # Normaliza o texto para o formato NFD e remove os caracteres diacríticos
        return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto

# Remover acentuação dos textos na coluna 'Nome da Mãe'
adendos_df['Nome da Mãe'] = adendos_df['Nome da Mãe'].apply(remover_acentuacao)
relatorios_df['Nome da Mãe'] = relatorios_df['Nome da Mãe'].apply(remover_acentuacao)
# Definir palavras a serem removidas
palavras_para_remover = r'\bgemelar\b|\bgemelar1\b|\bgemelar2\b|\bg1\b|\bg2\b'

# Remover as palavras específicas da coluna 'Nome da Mãe' em ambos os DataFrames
adendos_df['Nome da Mãe'] = adendos_df['Nome da Mãe'].str.replace(palavras_para_remover, '', regex=True)
relatorios_df['Nome da Mãe'] = relatorios_df['Nome da Mãe'].str.replace(palavras_para_remover, '', regex=True)


In [ ]:
# Remover qualquer numeração da coluna 'Nome da Mãe' em ambos os DataFrames
adendos_df['Nome da Mãe'] = adendos_df['Nome da Mãe'].str.replace(r'\d+', '', regex=True)
relatorios_df['Nome da Mãe'] = relatorios_df['Nome da Mãe'].str.replace(r'\d+', '', regex=True)

#Remover espaços em brancos
adendos_df['Nome da Mãe'] = adendos_df['Nome da Mãe'].str.replace(r'\s+', '', regex=True)
relatorios_df['Nome da Mãe'] = relatorios_df['Nome da Mãe'].str.replace(r'\s+', '', regex=True)

# Remover caracteres especiais como 'º' e 'ª' da coluna 'Nome da Mãe' em ambos os DataFrames
adendos_df['Nome da Mãe'] = adendos_df['Nome da Mãe'].str.replace(r'[ºª]', '', regex=True)
relatorios_df['Nome da Mãe'] = relatorios_df['Nome da Mãe'].str.replace(r'[ºª]', '', regex=True)


In [ ]:
print(adendos_df.shape,relatorios_df.shape)

Agora vou tirar os que não tem nome no arquivo, retirei algumas manualmente pois eram documentos sem nenhuma informação


In [ ]:
# Remover linhas com 'Nome do Arquivo' sendo NULL ou vazio em adendos_df
adendos_df = adendos_df.dropna(subset=['Nome do Arquivo'])
adendos_df = adendos_df[adendos_df['Nome do Arquivo'].str.strip() != '']

# Remover linhas com 'Nome do Arquivo' sendo NULL ou vazio em relatorios_df
relatorios_df = relatorios_df.dropna(subset=['Nome do Arquivo'])
relatorios_df = relatorios_df[relatorios_df['Nome do Arquivo'].str.strip() != '']

# Excluir linhas onde 'Nome da Mãe' é NaN ou vazio em adendos_df
adendos_df = adendos_df.dropna(subset=['Nome da Mãe'])
adendos_df = adendos_df[adendos_df['Nome da Mãe'].str.strip() != '']

# Excluir linhas onde 'Nome da Mãe' é NaN ou vazio em relatorios_df
relatorios_df = relatorios_df.dropna(subset=['Nome da Mãe'])
relatorios_df = relatorios_df[relatorios_df['Nome da Mãe'].str.strip() != '']



In [ ]:
# Contar quantas linhas têm "Nome da Mãe" como NaN ou string vazia em adendos_df
sem_nome_mae_adendos = adendos_df['Nome da Mãe'].isna().sum() + adendos_df['Nome da Mãe'].str.strip().eq('').sum()

# Contar quantas linhas têm "Nome da Mãe" como NaN ou string vazia em relatorios_df
sem_nome_mae_relatorios = relatorios_df['Nome da Mãe'].isna().sum() + relatorios_df['Nome da Mãe'].str.strip().eq('').sum()

# Exibir o resultado
print(f"Número de linhas sem 'Nome da Mãe' em adendos_df: {sem_nome_mae_adendos}")
print(f"Número de linhas sem 'Nome da Mãe' em relatorios_df: {sem_nome_mae_relatorios}")


In [ ]:
print(relatorios_df.shape,adendos_df.shape)

Organizando pelo Nome da Mãe

In [ ]:
adendos_df = adendos_df.sort_values(by='Nome da Mãe')
relatorios_df = relatorios_df.sort_values(by='Nome da Mãe')

print(relatorios_df.shape,adendos_df.shape)

In [ ]:
print(relatorios_df.shape,adendos_df.shape)

Vendo as linhas que estão repetidas "ou casos de gemeralidade"

In [ ]:
duplicados_adendos = adendos_df[adendos_df.duplicated(subset=['Nome da Mãe'], keep=False)]
duplicados_relatorios = relatorios_df[relatorios_df.duplicated(subset=['Nome da Mãe'], keep=False)]

duplicados_adendos

In [ ]:
print(relatorios_df.shape,adendos_df.shape)

Removendo as linhas duplicadas do dataframe original, depois mais a frente vou adiciona-las novamente condensadas

In [ ]:
# Remover linhas de adendos_df que estão em duplicados_adendos
adendos_df = adendos_df.drop(duplicados_adendos.index)

# Remover linhas de relatorios_df que estão em duplicados_relatorios
relatorios_df = relatorios_df.drop(duplicados_relatorios.index)

In [ ]:
print(relatorios_df.shape,adendos_df.shape)

In [ ]:
def condensar_valores(col):
    return ', '.join(col.dropna().unique())

# Condensar duplicados em adendos_df
adendos_condensado = duplicados_adendos.groupby('Nome da Mãe').agg(condensar_valores).reset_index()

# Condensar duplicados em relatorios_df
relatorios_condensado = duplicados_relatorios.groupby('Nome da Mãe').agg(condensar_valores).reset_index()

print(relatorios_condensado.shape,adendos_condensado.shape)

In [ ]:
relatorios_condensado

Agora vou acrescentar as linhas condensadas

In [ ]:
# Concatenar adendos_df com adendos_condensado
adendos_df = pd.concat([adendos_df, adendos_condensado], ignore_index=True)

# Concatenar relatorios_df com relatorios_condensado
relatorios_df = pd.concat([relatorios_df, relatorios_condensado], ignore_index=True)

print(relatorios_df.shape,adendos_df.shape)

Só verificando que tanto a linhas adendos_df quanto relatorios_df tem a mesma quantidade de nomes

In [ ]:
print(adendos_df['Nome da Mãe'].unique().shape, adendos_df.shape)
print(relatorios_df['Nome da Mãe'].unique().shape, relatorios_df.shape)

In [ ]:
# Verificar duplicatas em adendos_df com base em 'Data de Nascimento' e 'Peso'
duplicatas_adendos = adendos_df[adendos_df.duplicated(subset=['Data de Nascimento', 'Peso'], keep=False)]

# Exibir as duplicatas encontradas
print("Duplicatas em adendos_df com a mesma Data de Nascimento e Peso:")
print(duplicatas_adendos)

# Verificar duplicatas em relatorios_df com base em 'Data de Nascimento' e 'Peso'
duplicatas_relatorios = relatorios_df[relatorios_df.duplicated(subset=['Data de Nascimento', 'Peso'], keep=False)]

# Exibir as duplicatas encontradas
print("Duplicatas em relatorios_df com a mesma Data de Nascimento e Peso:")
print(duplicatas_relatorios)

Retirando os que tem a mesma data de nascimento e peso ao mesmo tempo

In [ ]:
adendos_df = adendos_df[~adendos_df.index.isin(duplicatas_adendos.index)]
relatorios_df = relatorios_df[~relatorios_df.index.isin(duplicatas_relatorios.index)]
print(relatorios_df.shape,adendos_df.shape)

In [ ]:
def combinar_linhas(grupo):
    # Combinar os dados, mantendo todas as informações distintas
    return grupo.groupby(['Data de Nascimento', 'Peso'], as_index=False).agg(lambda x: ','.join(x.astype(str).unique()))

# Aplicar a função de combinação ao DataFrame
adendos_condensado = combinar_linhas(duplicatas_adendos)
relatorios_condensado = combinar_linhas(duplicatas_relatorios)

# Exibir o DataFrame condensado
relatorios_condensado

In [ ]:
adendos_df = pd.concat([adendos_df, adendos_condensado], ignore_index=True)
relatorios_df = pd.concat([relatorios_df, relatorios_condensado], ignore_index=True)
print(relatorios_df.shape,adendos_df.shape)

##Agora vamos estudar as coisas de um conjunto que esta no outro e vice-versa

Separando os que tem o mesmo nome nos dois conjuntos de dados

In [ ]:
# Obter os nomes únicos da coluna 'Nome da Mãe' de cada DataFrame
nomes_adendos = set(adendos_df['Nome da Mãe'].unique())
nomes_relatorios = set(relatorios_df['Nome da Mãe'].unique())

# Encontrar nomes que estão em adendos_df mas não em relatorios_df
nomes_somente_adendos = nomes_adendos - nomes_relatorios

# Encontrar nomes que estão em relatorios_df mas não em adendos_df
nomes_somente_relatorios = nomes_relatorios - nomes_adendos

# Filtrar as linhas do DataFrame adendos_df com nomes presentes apenas em adendos_df
linhas_somente_adendos = adendos_df[adendos_df['Nome da Mãe'].isin(nomes_somente_adendos)]
adendos_df = adendos_df[~adendos_df.index.isin(linhas_somente_adendos.index)]

# Filtrar as linhas do DataFrame relatorios_df com nomes presentes apenas em relatorios_df
linhas_somente_relatorios = relatorios_df[relatorios_df['Nome da Mãe'].isin(nomes_somente_relatorios)]
relatorios_df = relatorios_df[~relatorios_df.index.isin(linhas_somente_relatorios.index)]

# Exibir os resultados

print(linhas_somente_relatorios.shape,linhas_somente_adendos.shape, adendos_df.shape, relatorios_df.shape)


Juntado agora os adendos_df e relatorios_df, e concatenando

In [ ]:
juntado_df = pd.concat([adendos_df, relatorios_df], ignore_index=True)

def combinar_linhas(grupo):
    # Combinar os dados, mantendo todas as informações distintas
    return grupo.agg(lambda x: ','.join(x.dropna().astype(str).unique()))

# Agrupar por 'Nome da Mãe' e aplicar a função de combinação
juntado_condensado = juntado_df.groupby('Nome da Mãe', as_index=False).apply(combinar_linhas)

# Resetar o índice para garantir que o DataFrame esteja corretamente formatado
juntado_condensado.reset_index(drop=True, inplace=True)

# Exibir o DataFrame condensado
print("DataFrame condensado:")
print(juntado_condensado)

In [ ]:
print(linhas_somente_adendos.shape,linhas_somente_relatorios.shape)

In [ ]:
import pandas as pd

# Extrair pares únicos de 'Data de Nascimento' e 'Peso'
pares_adendos = set(linhas_somente_adendos[['Data de Nascimento', 'Peso']].itertuples(index=False, name=None))
pares_relatorios = set(linhas_somente_relatorios[['Data de Nascimento', 'Peso']].itertuples(index=False, name=None))

# Encontrar pares comuns
pares_comuns = pares_adendos.intersection(pares_relatorios)

# Filtrar linhas correspondentes em 'linhas_somente_adendos'
linhas_correspondentes_adendos = linhas_somente_adendos[
    linhas_somente_adendos[['Data de Nascimento', 'Peso']].apply(tuple, axis=1).isin(pares_comuns)
]

# Filtrar linhas correspondentes em 'linhas_somente_relatorios'
linhas_correspondentes_relatorios = linhas_somente_relatorios[
    linhas_somente_relatorios[['Data de Nascimento', 'Peso']].apply(tuple, axis=1).isin(pares_comuns)
]

# Encontrar linhas em 'linhas_somente_adendos' que não estão em 'linhas_correspondentes_adendos'
linhas_nao_correspondentes_adendos = linhas_somente_adendos[
    ~linhas_somente_adendos[['Data de Nascimento', 'Peso']].apply(tuple, axis=1).isin(pares_comuns)
]

# Encontrar linhas em 'linhas_somente_relatorios' que não estão em 'linhas_correspondentes_relatorios'
linhas_nao_correspondentes_relatorios = linhas_somente_relatorios[
    ~linhas_somente_relatorios[['Data de Nascimento', 'Peso']].apply(tuple, axis=1).isin(pares_comuns)
]

# Exibir os resultados
print("Linhas em 'linhas_somente_adendos' que não têm correspondência em 'linhas_somente_relatorios':")
print(linhas_nao_correspondentes_adendos)

print("\nLinhas em 'linhas_somente_relatorios' que não têm correspondência em 'linhas_somente_adendos':")
print(linhas_nao_correspondentes_relatorios)


In [ ]:
print(linhas_correspondentes_adendos.shape,linhas_correspondentes_relatorios.shape, linhas_nao_correspondentes_adendos.shape, linhas_nao_correspondentes_relatorios.shape)

In [ ]:
linhas_correspondentes_combinadas = pd.concat([linhas_correspondentes_adendos, linhas_correspondentes_relatorios], ignore_index=True)

def combinar_linhas(grupo):
    # Combinar os dados, mantendo todas as informações distintas
    return grupo.groupby(['Data de Nascimento', 'Peso'], as_index=False).agg(lambda x: ','.join(x.astype(str).unique()))

# Aplicar a função de combinação ao DataFrame
linhas_correspondentes_combinadas = combinar_linhas(linhas_correspondentes_combinadas)


In [ ]:
linhas_correspondentes_combinadas.shape

In [ ]:
def combinar_colunas(series):
    # Combinar valores distintos em uma única string separada por vírgula
    return ','.join(series.dropna().astype(str).unique())

# Mesclar os DataFrames com base nas colunas 'Data de Nascimento' e 'Peso'
merged_df = pd.merge(linhas_correspondentes_adendos, linhas_correspondentes_relatorios,
                     on=['Data de Nascimento', 'Peso'],
                     how='outer',
                     suffixes=('_adendos', '_relatorios'))

# Agrupar por 'Data de Nascimento' e 'Peso', combinando as colunas com valores distintos
resultado_df = merged_df.groupby(['Data de Nascimento', 'Peso'], as_index=False).agg(combinar_colunas)

# Exibir o DataFrame resultante
print("DataFrame mesclado e combinado:")
print(resultado_df)

In [ ]:
todos_df = pd.concat([resultado_df,
                      linhas_nao_correspondentes_relatorios,
                      linhas_nao_correspondentes_adendos,
                      juntado_condensado], ignore_index=True)


##Salvando no Excel

In [ ]:
!pip install openpyxl
caminho_arquivo = '/content/resultado_dados.xlsx'

# Criar um objeto ExcelWriter para escrever os DataFrames em abas diferentes
with pd.ExcelWriter(caminho_arquivo, engine='openpyxl') as writer:
    # Escrever cada DataFrame em uma aba diferente
    resultado_df.to_excel(writer, sheet_name='Resultado', index=False)
    linhas_nao_correspondentes_relatorios.to_excel(writer, sheet_name='Linhas_Nao_Correspondentes_Relatorios', index=False)
    linhas_nao_correspondentes_adendos.to_excel(writer, sheet_name='Linhas_Nao_Correspondentes_Adendos', index=False)
    juntado_condensado.to_excel(writer, sheet_name='Juntado_Condensado', index=False)

print(f"DataFrames salvos em {caminho_arquivo}")

##Dei uma limpada manualmente nos dados e vamos subir novamente

In [ ]:
arquivo_excel = '/content/drive/MyDrive/PRIVATE_DATA/Cópia de Dados_Adendos_Relatorios_sem_match_03_09.xlsx'
xls = pd.ExcelFile(arquivo_excel)
print(xls.sheet_names)



In [ ]:
linhas_nao_correspondentes_adendos = pd.read_excel(arquivo_excel, sheet_name='lin_ad')
linhas_nao_correspondentes_relatorios = pd.read_excel(arquivo_excel, sheet_name='lin_rel')

In [ ]:
print(linhas_nao_correspondentes_adendos.shape, linhas_nao_correspondentes_relatorios.shape)

In [ ]:
# Supondo que os DataFrames sejam chamados de 'linhas_nao_correspondentes_relatorios' e 'linhas_nao_correspondentes_adendos'
# E que a coluna com o nome da mãe seja chamada 'Nome_Mae'

# Obter os nomes únicos da coluna 'Nome_Mae' de cada DataFrame
nomes_adendos = set(linhas_nao_correspondentes_adendos['Nome da Mãe'].unique())
nomes_relatorios = set(linhas_nao_correspondentes_relatorios['Nome da Mãe'].unique())

# Encontrar nomes que estão em linhas_nao_correspondentes_adendos mas não em linhas_nao_correspondentes_relatorios
nomes_somente_adendos = nomes_adendos - nomes_relatorios

# Encontrar nomes que estão em linhas_nao_correspondentes_relatorios mas não em linhas_nao_correspondentes_adendos
nomes_somente_relatorios = nomes_relatorios - nomes_adendos

# Filtrar as linhas do DataFrame linhas_nao_correspondentes_adendos com nomes presentes apenas em adendos
linhas_somente_adendos = linhas_nao_correspondentes_adendos[linhas_nao_correspondentes_adendos['Nome da Mãe'].isin(nomes_somente_adendos)]

# Atualizar linhas_nao_correspondentes_adendos removendo as linhas encontradas
linhas_nao_correspondentes_adendos = linhas_nao_correspondentes_adendos[~linhas_nao_correspondentes_adendos.index.isin(linhas_somente_adendos.index)]

# Filtrar as linhas do DataFrame linhas_nao_correspondentes_relatorios com nomes presentes apenas em relatorios
linhas_somente_relatorios = linhas_nao_correspondentes_relatorios[linhas_nao_correspondentes_relatorios['Nome da Mãe'].isin(nomes_somente_relatorios)]

# Atualizar linhas_nao_correspondentes_relatorios removendo as linhas encontradas
linhas_nao_correspondentes_relatorios = linhas_nao_correspondentes_relatorios[~linhas_nao_correspondentes_relatorios.index.isin(linhas_somente_relatorios.index)]

# Exibir os resultados
print("Linhas apenas em Relatórios:", linhas_somente_relatorios.shape)
print("Linhas apenas em Adendos:", linhas_somente_adendos.shape)
print("Adendos após remoção:", linhas_nao_correspondentes_adendos.shape)
print("Relatórios após remoção:", linhas_nao_correspondentes_relatorios.shape)


In [ ]:
print(linhas_nao_correspondentes_adendos["Nome da Mãe"], linhas_nao_correspondentes_relatorios['Nome da Mãe'])

In [ ]:
print(linhas_nao_correspondentes_relatorios["Nome da Mãe"], linhas_nao_correspondentes_adendos["Nome da Mãe"])

# Quem tem Adendo e não tem relatório será que morreu?

In [ ]:
linhas_somente_relatorios

In [ ]:
linhas_somente_adendos

In [ ]:
# Caminho para o arquivo Excel no Google Drive
file_path = '/content/drive/MyDrive/PRIVATE_DATA/Cópia de LISTA DE ÓBITOS PRECOCES E TARDIOS 2019-2023.xlsx'

# Ler a planilha do Excel como um DataFrame
obito_df = pd.read_excel(file_path)

In [ ]:
obito_df['Nome Paciente'] = obito_df['Nome Paciente'].str.replace(r'^RN\s*', '', regex=True)
obito_df['Nome Paciente'] = obito_df['Nome Paciente'].str.replace(r'^ RN\s*', '', regex=True)
obito_df['Nome Paciente'] = obito_df['Nome Paciente'].str.replace(r' ', '', regex=True)
obito_df

In [ ]:
!pip install unidecode

import unidecode

def normalizar_texto(texto):
    texto = texto.lower()  # Converter para minúsculas
    texto = unidecode.unidecode(texto)  # Remover acentos e caracteres especiais
    return texto

# Aplicar a função de normalização na coluna 'Nome Paciente'
obito_df['Nome Paciente'] = obito_df['Nome Paciente'].apply(normalizar_texto)

In [ ]:
obito_df

In [ ]:
resultado_merge = pd.merge(
    obito_df,
    linhas_somente_adendos,
    left_on='Nome Paciente',
    right_on='Nome da Mãe',
    how='inner'
)

resultado_merge

In [ ]:
obito_df_sem_merge = obito_df[~obito_df['Nome Paciente'].isin(resultado_merge['Nome Paciente'])]

# Para encontrar as linhas em 'linhas_somente_adendos' que foram incluídas no merge
linhas_somente_adendos_sem_merge = linhas_somente_adendos[~linhas_somente_adendos['Nome da Mãe'].isin(resultado_merge['Nome da Mãe'])]

print(obito_df_sem_merge.shape,linhas_somente_adendos_sem_merge.shape)

In [ ]:
obito_df_sem_merge

In [ ]:
obito_df_sem_merge['Dt. Nasc'] = pd.to_datetime(obito_df_sem_merge['Dt. Nasc'], errors='coerce')
linhas_somente_adendos_sem_merge['Data de Nascimento'] = pd.to_datetime(linhas_somente_adendos_sem_merge['Data de Nascimento'], errors='coerce')

resultado_data_nascimento = pd.merge(
    obito_df_sem_merge,
    linhas_somente_adendos_sem_merge,
    left_on='Dt. Nasc',
    right_on='Data de Nascimento',
    how='inner'
)

In [ ]:
resultado_data_nascimento

In [ ]:
!pip install Levenshtein
import Levenshtein as lev
def calcular_similaridade(nome1, nome2):
    return lev.ratio(nome1, nome2)

# Aplicar a função de similaridade para cada linha do DataFrame
resultado_data_nascimento['Similaridade Levenshtein'] = resultado_data_nascimento.apply(
    lambda row: calcular_similaridade(row['Nome Paciente'], row['Nome da Mãe']),
    axis=1
)

# Exibir as primeiras linhas do DataFrame resultante para verificar
resultado_data_nascimento

In [ ]:
file_path = '/content/drive/MyDrive/PRIVATE_DATA/resultado_data_nascimento.xlsx'
existing_data = pd.read_excel(file_path, sheet_name=None)  # Carrega todas as abas

# Adicionar as novas abas
existing_data['linhas_somente_adendos'] = linhas_somente_adendos
existing_data['linhas_somente_relatorios'] = linhas_somente_relatorios

# Salvar o arquivo com as novas abas
with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
    for sheet_name, df in existing_data.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Novas abas foram adicionadas com sucesso.")

In [ ]:
output_file_path = '/content/drive/MyDrive/PRIVATE_DATA/resultado_data_nascimento.xlsx'

# Salvar o DataFrame em um arquivo Excel
resultado_data_nascimento.to_excel(output_file_path, index=False)

print(f'O arquivo foi salvo em: {output_file_path}')

##Linkage

Subindo os RNV/Natimortos

In [ ]:
relatorio_nascidos_vivos = '/content/drive/MyDrive/PRIVATE_DATA/RNV_e_Natimorto_Total.xlsx'

df_combined = pd.read_excel(relatorio_nascidos_vivos)

In [ ]:
df_combined = df_combined.sort_values(by='NºDNV')
df_combined

In [ ]:
df_combined = df_combined.dropna(subset=['NOME'])
df_combined

In [ ]:
!pip install unidecode
import unidecode

In [ ]:
df_combined['NOME'] = df_combined['NOME'].str.lower().str.replace(' ', '').str.replace('ç', 'c').apply(unidecode.unidecode)
df_combined['NOME'] = df_combined['NOME'].str.replace(r'g[0-9]+$', '', regex=True)
df_combined['NOME'] = df_combined['NOME'].str.replace(r'gemelar[0-9]+$', '', regex=True)

df_combined

In [ ]:
df_combined['GEMELAR'] = df_combined.duplicated(subset=['NOME'], keep=False).astype(int)

df_combined

In [ ]:
df_combined = df_combined[~df_combined['NOME'].str.contains('cancelada|cancelado|desconhecida|desconhecido', case=False)]
df_combined

In [ ]:
df_combined['DATA NASC.'] = pd.to_datetime(df_combined['DATA NASC.'], format='%d/%m/%Y', errors='coerce')
df_combined = df_combined.dropna(subset=['DATA NASC.'])
df_combined['DATA NASC.'] = df_combined['DATA NASC.'].dt.strftime('%d/%m/%Y')

df_combined

##Linkage de Fato

In [ ]:
df_notas_clinicas = todos_df.copy()
df_notas_clinicas

In [ ]:
df_merged = pd.merge(df_combined, df_notas_clinicas, left_on='NOME', right_on='Nome da Mãe', how='inner')

df_merged

In [ ]:
mask = ~df_notas_clinicas['Nome da Mãe'].isin(df_merged['Nome da Mãe'])

# Aplique a máscara ao DataFrame df_notas_clinicas
df_notas_clinicas_filtered = df_notas_clinicas[mask]

df_notas_clinicas_filtered

In [ ]:
df_notas_clinicas_filtered.rename(columns={'Nome da Mãe': 'NOME', 'Data de Nascimento': 'DATA NASC.', 'Peso': 'PESO'}, inplace=True)

df_combined['DATA NASC.'] = pd.to_datetime(df_combined['DATA NASC.'], errors='coerce')
df_notas_clinicas_filtered['DATA NASC.'] = pd.to_datetime(df_notas_clinicas_filtered['DATA NASC.'], errors='coerce')

In [ ]:
# Converter as colunas 'NOME' para string em ambos os DataFrames
df_combined['NOME'] = df_combined['NOME'].astype(str)
df_notas_clinicas_filtered['NOME'] = df_notas_clinicas_filtered['NOME'].astype(str)

# Verificar novamente os tipos de dados
print(df_combined['NOME'].dtype)
print(df_notas_clinicas_filtered['NOME'].dtype)

In [ ]:
df_combined['PESO'] = pd.to_numeric(df_combined['PESO'], errors='coerce')

In [ ]:
!pip install recordlinkage
import recordlinkage
from recordlinkage.index import Full

In [ ]:
df_notas_clinicas_filtered['PESO'] = pd.to_numeric(df_notas_clinicas_filtered['PESO'], errors='coerce')


In [ ]:
df_notas_clinicas_filtered

In [ ]:
df_combined['PESO'] = df_combined['PESO'].astype(str)


In [ ]:
indexer = recordlinkage.Index()
indexer.block('DATA NASC.')

# Gerar os links candidatos
candidate_links = indexer.index(df_combined, df_notas_clinicas_filtered)

# Cria um objeto comparador
comparer = recordlinkage.Compare()

# Adiciona comparações para as colunas específicas
comparer.string('NOME', 'NOME', method='jarowinkler', threshold=0.90, label='nome')
comparer.date('DATA NASC.', 'DATA NASC.', label='data_nasc')
comparer.numeric('PESO', 'PESO', label='peso')
features = comparer.compute(candidate_links, df_combined, df_notas_clinicas_filtered)
matches = features[features.sum(axis=1) > 2]
indices_df_combined, indices_df_notas_clinicas = matches.index.get_level_values(0), matches.index.get_level_values(1)
matched_records_df_combined = df_combined.loc[indices_df_combined]
matched_records_df_notas_clinicas = df_notas_clinicas.loc[indices_df_notas_clinicas]

# Resetando o índice para facilitar a concatenação
matched_records_df_combined.reset_index(drop=True, inplace=True)
matched_records_df_notas_clinicas.reset_index(drop=True, inplace=True)

# Concatenando os DataFrames lado a lado para visualização
matched_df_1 = pd.concat([matched_records_df_combined, matched_records_df_notas_clinicas], axis=1)
matched_df_1

In [ ]:
print(df_combined.dtypes)
print(df_notas_clinicas_filtered.dtypes)